# RACE RC Quiz System — Colab / Kaggle Runner

End-to-end runner for the traditional-ML pipeline.

**Tested on:** Google Colab (Python 3.11, T4 GPU) and Kaggle (Python 3.11, P100/T4 GPU).

## What runs on GPU?
Only **XGBoost** uses CUDA (auto-detected). All other models (LR, SVM, NB, RF, KMeans, Label Propagation) are sklearn → CPU only. The bigger speedup vs your laptop comes from more vCPUs and RAM.

## Steps
1. (Once) Set Runtime → GPU (Colab) or Settings → Accelerator → GPU (Kaggle).
2. Run cells top-to-bottom.
3. After the LAST cell finishes, download `models/` and `reports/metrics_test.json` (Colab) or use Kaggle's output panel.

## 1. Clone the repository

Replace the URL below with your fork if needed.

In [ ]:
import os, sys, subprocess, pathlib

REPO_URL  = 'https://github.com/SydBurhan/race_rc_project-.git'  # edit if needed
REPO_NAME = 'race_rc_project-'

if not pathlib.Path(REPO_NAME).exists():
    subprocess.run(['git', 'clone', '--depth=1', REPO_URL], check=True)
os.chdir(REPO_NAME)
print('cwd =', os.getcwd())
print(os.listdir('.'))

## 2. Install dependencies (incl. gensim — works on Python 3.11)

In [ ]:
!pip install -q -r requirements.txt 'gensim>=4.3.0'
import nltk
for pkg in ['stopwords', 'punkt', 'punkt_tab', 'wordnet', 'omw-1.4']:
    nltk.download(pkg, quiet=True)
print('Deps OK')

## 3. Verify GPU is visible to XGBoost
If this prints `cuda`, RF stays on CPU but XGBoost will be 5–10x faster.

In [ ]:
import subprocess
try:
    print(subprocess.check_output(['nvidia-smi', '-L'], text=True))
except Exception as e:
    print('No GPU detected:', e)

from src.model_a_train import _detect_xgb_device
print('XGBoost device:', _detect_xgb_device())

## 4. Get the RACE dataset

Pick **one** of the options below.

### Option A — Kaggle: attach `ankitdhiman7/race-dataset` via the right-hand panel.
On Kaggle, datasets land at `/kaggle/input/race-dataset/`. The cell below copies the CSVs into `data/raw/`.

### Option B — Colab: upload your own CSVs (or `kaggle.json`) and download.
Easiest: drag-drop `train.csv` (and optionally `dev.csv` / `test.csv`) into the file panel, then move them into `data/raw/`.

### Option C — HuggingFace (true splits, no duplicates).
Uses `datasets` to grab the canonical RACE splits.

In [ ]:
# === OPTION A: Kaggle attached dataset ===
import os, shutil, glob, pathlib
pathlib.Path('data/raw').mkdir(parents=True, exist_ok=True)
kaggle_root = '/kaggle/input'
if os.path.isdir(kaggle_root):
    for csv in glob.glob(f'{kaggle_root}/**/*.csv', recursive=True):
        name = os.path.basename(csv).lower()
        if name in {'train.csv', 'dev.csv', 'val.csv', 'test.csv', 'validation.csv'}:
            shutil.copy(csv, 'data/raw/' + name)
            print('copied', csv)
print('data/raw =', os.listdir('data/raw'))

In [ ]:
# === OPTION C: HuggingFace canonical RACE (run only if you didn't use A or B) ===
# Uncomment this cell if you want true splits.
#
# !pip install -q datasets
# import pandas as pd, pathlib
# from datasets import load_dataset
# pathlib.Path('data/raw').mkdir(parents=True, exist_ok=True)
# ds = load_dataset('ehovy/race', 'all')
# def to_df(split_ds):
#     rows = []
#     for r in split_ds:
#         opts = r['options']
#         rows.append({'id': r.get('example_id', ''),
#                      'article': r['article'], 'question': r['question'],
#                      'A': opts[0], 'B': opts[1], 'C': opts[2], 'D': opts[3],
#                      'answer': r['answer']})
#     return pd.DataFrame(rows)
# to_df(ds['train']).to_csv('data/raw/train.csv', index=False)
# to_df(ds['validation']).to_csv('data/raw/val.csv', index=False)
# to_df(ds['test']).to_csv('data/raw/test.csv', index=False)
# print('Saved real splits')

## 5. Preprocessing

Builds OHE + lexical + TF-IDF features and saves processed splits.

In [ ]:
!python src/preprocessing.py

## 6. Train Model A (LR + SVM + NB + RF + XGBoost + ensemble + KMeans + LabelProp)

In [ ]:
!python src/model_a_train.py

## 7. Train question-ranker (template generator)

In [ ]:
!python src/template_generator.py

## 8. Train Model B (hint scorer + Word2Vec cache)

In [ ]:
# Word2Vec download is ~1.6 GB. Comment '--skip-w2v' off to keep it lightweight.
!python src/model_b_train.py

## 9. NLP Evaluation (BLEU / ROUGE-L / METEOR on 500 test rows)

In [ ]:
!python src/evaluate_nlp.py --max 500

## 10. Show the final metrics file

In [ ]:
import json
print(json.dumps(json.load(open('reports/metrics_test.json')), indent=2))

## 11. (Optional) Run the Streamlit UI from Colab via ngrok

Skip this on Kaggle — Kaggle blocks tunnels. Use it only on Colab.

In [ ]:
# !pip install -q pyngrok
# from pyngrok import ngrok
# # Set your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
# # ngrok.set_auth_token('YOUR_TOKEN')
# import subprocess
# proc = subprocess.Popen(['streamlit', 'run', 'ui/app.py',
#                           '--server.port', '8501', '--server.headless', 'true'])
# print('Public URL:', ngrok.connect(8501).public_url)

## 12. (Optional) Bundle artefacts for download

Creates a single `race_artifacts.zip` containing trained models and reports — easy to download in one click.

In [ ]:
import shutil
shutil.make_archive('race_artifacts', 'zip', root_dir='.',
                     base_dir='models')
shutil.make_archive('race_reports', 'zip', root_dir='.',
                     base_dir='reports')
print('Created race_artifacts.zip and race_reports.zip')